# ABO-150 Expanded: VLM Validation (Colab)

Этот ноутбук:
1. Ставит зависимости.
2. Клонирует приватный репозиторий.
3. Чинит `panel_path` в `selected_150_annotations.jsonl`.
4. Валидирует `dataset/abo_150_expanded` по ontology из `physics_properties.yaml`.
5. Сохраняет отчёты и логирует в Comet (как в `CourseWork.ipynb`).

Colab Secrets:
- `git_coursework` (GitHub token для приватного репо)
- `comet_api_key` (опционально)
- `comet_workspace` (опционально)
- `comet_project_name` (опционально)

In [ ]:
# Dependencies (Colab-safe versions + one-time auto-restart)
import os
import sys
import subprocess
from pathlib import Path

MARKER = Path('/tmp/vlm_abo150_validation_deps_ready')

if not MARKER.exists():
    print('Installing dependencies (first run)...')

    install_cmds = [
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'numpy==2.1.3'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'transformers>=4.49.0,<5.0.0', 'accelerate', 'bitsandbytes', 'sentencepiece'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'pandas==2.2.2', 'pillow<12', 'pyyaml', 'boto3', 'rembg', 'onnxruntime', 'comet_ml'],
    ]

    for cmd in install_cmds:
        print('>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime now...')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this runtime. Continue.')

In [ ]:
# Clone or update private repo in Colab
import os
import subprocess
from pathlib import Path

from google.colab import userdata

REPO_SLUG = "Yaitco/VLM-2D-Physics-Boundaries"
WORKDIR = Path("/content/VLM-2D-Physics-Boundaries")

GITHUB_TOKEN = userdata.get('git_coursework')
if not GITHUB_TOKEN:
    raise ValueError("Colab secret 'git_coursework' is missing. Add it in Secrets and rerun this cell.")

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git"

if not WORKDIR.exists():
    subprocess.run(["git", "clone", clone_url, str(WORKDIR)], check=True)
else:
    print(f"Repo already exists: {WORKDIR}. Pulling latest...")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)

%cd /content/VLM-2D-Physics-Boundaries

In [ ]:
# Fix panel_path in ABO-150 annotations
!python scripts/update_abo150_panel_paths.py   --annotations-path dataset/abo_150_expanded/selected_150_annotations.jsonl   --panels-dir dataset/abo_150_expanded/selected_150_photos/panels   --path-mode relative

# Quick check
import json
from pathlib import Path

ann_path = Path('dataset/abo_150_expanded/selected_150_annotations.jsonl')
first = json.loads(next(ann_path.open('r', encoding='utf-8')))
print('Example panel_path:', first['panels']['panel_path'])

In [ ]:
import json
from pathlib import Path

import pandas as pd

from scripts.abo150_vlm_validation import (
    MODEL_REGISTRY,
    get_available_variants,
    init_comet_experiment,
    is_known_value,
    load_abo150_samples,
    load_protocol_property_specs,
    run_many_models,
    run_validation,
    save_report,
)

import inspect

def _call_with_supported_kwargs(fn, **kwargs):
    sig = inspect.signature(fn)
    supported = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return fn(**supported)


def run_validation_safe(**kwargs):
    return _call_with_supported_kwargs(run_validation, **kwargs)


def run_many_models_safe(**kwargs):
    return _call_with_supported_kwargs(run_many_models, **kwargs)


# ---------- Dataset ----------
DATASET_NAME = "abo_150_expanded"
DATASET_DIR = Path("dataset") / DATASET_NAME
ANNOTATIONS_PATH = DATASET_DIR / "selected_150_annotations.jsonl"
PHYSICS_SCHEMA_PATH = DATASET_DIR / "physics_properties.yaml"
REPORTS_DIR = Path("reports_abo150_expanded")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Evaluation protocol ----------
PROTOCOL_NAME = "narrow_core"  # narrow_core | expanded_ontology | full_expanded | pdf_compact

# ---------- Model ----------
SELECTED_MODEL = "qwen2_5_vl_7b"
assert SELECTED_MODEL in MODEL_REGISTRY, f"Unknown model key: {SELECTED_MODEL}"

# ---------- Evaluation config ----------
MAX_SAMPLES = None
RANDOM_SEED = 42
EVAL_VARIANTS = ["raw"]  # simplest baseline: raw image only
MASK_BACKGROUND_MODE = "black"
SAVE_RAW_OUTPUT = True

# Prompt shaping
INCLUDE_ONLY_GT_KNOWN = False  # run all properties from the selected protocol
MAX_PROPERTIES_PER_SAMPLE = None  # set int (e.g., 24/32) if prompts become too long
PROMPT_MODE = "per_property"  # per_property | grouped | joint
PROPERTY_BATCH_SIZE = 8  # how many prompts to send in one batched generate
PROPERTY_GROUP_SIZE = 4  # used only in grouped mode: how many properties go into one prompt

# Few-shot
FEW_SHOT_K = 0  # simplest baseline: zero-shot
FEW_SHOT_SELECTION_MODE = "fixed"  # fixed | dynamic

# Success threshold for has_valid_json / image_id_matched (mainly useful in per_property)
JSON_SUCCESS_THRESHOLD = 1.0
IMAGE_ID_SUCCESS_THRESHOLD = None

# Safety: joint mode with too many properties often leads to truncated JSON
if PROMPT_MODE == "joint" and MAX_PROPERTIES_PER_SAMPLE is None:
    MAX_PROPERTIES_PER_SAMPLE = 24 if PROTOCOL_NAME in {"pdf_compact", "narrow_core"} else 32
    print(f"PROMPT_MODE=joint: MAX_PROPERTIES_PER_SAMPLE auto-set to {MAX_PROPERTIES_PER_SAMPLE}.")

# ---------- Comet ----------
COMET_ENABLED = True
COMET_DEFAULT_PROJECT = "vlm-physics-validation"


In [ ]:
property_specs = load_protocol_property_specs(
    protocol_name=PROTOCOL_NAME,
    schema_path=PHYSICS_SCHEMA_PATH if PROTOCOL_NAME in {"expanded_ontology", "full_expanded", "narrow_core"} else None,
)
samples = load_abo150_samples(
    annotations_path=ANNOTATIONS_PATH,
    dataset_dir=DATASET_DIR,
    property_specs=property_specs,
    protocol_name=PROTOCOL_NAME,
    max_samples=MAX_SAMPLES,
    random_seed=RANDOM_SEED,
)

print(f"Loaded samples: {len(samples)}")
print(f"Protocol: {PROTOCOL_NAME}")
print(f"Evaluable properties: {len(property_specs)}")
print("First sample:")
print(json.dumps(
    {
        "image_id": samples[0]["image_id"],
        "path": samples[0]["path"],
        "known_properties": sum(
            1 for k, v in samples[0]["gt_properties"].items()
            if is_known_value(property_specs[k], v)
        ),
        "available_gt_keys": [
            k for k, v in samples[0]["gt_properties"].items()
            if is_known_value(property_specs[k], v)
        ][:10],
    },
    ensure_ascii=False,
    indent=2,
))


In [ ]:
available_variants = get_available_variants(EVAL_VARIANTS, samples)
print("Running variants:", available_variants)

run_params = {
    "dataset_name": DATASET_NAME,
    "annotations_path": str(ANNOTATIONS_PATH),
    "schema_path": str(PHYSICS_SCHEMA_PATH),
    "protocol_name": PROTOCOL_NAME,
    "selected_model_key": SELECTED_MODEL,
    "selected_model_id": MODEL_REGISTRY[SELECTED_MODEL]["model_id"],
    "eval_variants_requested": ",".join(EVAL_VARIANTS),
    "eval_variants_actual": ",".join(available_variants),
    "mask_background_mode": MASK_BACKGROUND_MODE,
    "max_samples": MAX_SAMPLES,
    "random_seed": RANDOM_SEED,
    "include_only_gt_known": INCLUDE_ONLY_GT_KNOWN,
    "max_properties_per_sample": MAX_PROPERTIES_PER_SAMPLE,
    "prompt_mode": PROMPT_MODE,
    "property_batch_size": PROPERTY_BATCH_SIZE,
    "property_group_size": PROPERTY_GROUP_SIZE,
    "few_shot_k": FEW_SHOT_K,
    "few_shot_selection_mode": FEW_SHOT_SELECTION_MODE,
    "json_success_threshold": JSON_SUCCESS_THRESHOLD,
    "image_id_success_threshold": IMAGE_ID_SUCCESS_THRESHOLD,
    "num_properties": len(property_specs),
}

comet_experiment = init_comet_experiment(
    run_tag=f"{SELECTED_MODEL}_{DATASET_NAME}_{PROTOCOL_NAME}",
    run_params=run_params,
    enabled=COMET_ENABLED,
    default_project=COMET_DEFAULT_PROJECT,
)

try:
    variant_metrics = []

    for variant in available_variants:
        df_variant = run_validation_safe(
            model_key=SELECTED_MODEL,
            samples=samples,
            property_specs=property_specs,
            model_registry=MODEL_REGISTRY,
            variant=variant,
            prompt_mode=PROMPT_MODE,
            property_batch_size=PROPERTY_BATCH_SIZE,
            property_group_size=PROPERTY_GROUP_SIZE,
            few_shot_k=FEW_SHOT_K,
            few_shot_selection_mode=FEW_SHOT_SELECTION_MODE,
            json_success_threshold=JSON_SUCCESS_THRESHOLD,
            image_id_success_threshold=IMAGE_ID_SUCCESS_THRESHOLD,
            include_only_gt_known=INCLUDE_ONLY_GT_KNOWN,
            max_properties_per_sample=MAX_PROPERTIES_PER_SAMPLE,
            mask_background_mode=MASK_BACKGROUND_MODE,
            save_raw_output=SAVE_RAW_OUTPUT,
        )

        pm_variant, summary_variant = save_report(
            df=df_variant,
            model_key=SELECTED_MODEL,
            variant=variant,
            model_registry=MODEL_REGISTRY,
            property_specs=property_specs,
            reports_dir=REPORTS_DIR / PROTOCOL_NAME,
            comet_experiment=comet_experiment,
        )

        pm_variant = pm_variant.copy()
        pm_variant["variant"] = variant
        variant_metrics.append(pm_variant)

    if variant_metrics:
        all_variant_metrics_df = pd.concat(variant_metrics, ignore_index=True)
        display(all_variant_metrics_df)

        if set(available_variants) >= {"raw", "masked"}:
            pivot = all_variant_metrics_df.pivot(index="property", columns="variant", values="coverage_on_gt_known_pct")
            if "raw" in pivot.columns and "masked" in pivot.columns:
                pivot["delta_masked_minus_raw"] = pivot["masked"] - pivot["raw"]
            print("\nCoverage comparison (masked vs raw):")
            display(pivot)

            if comet_experiment is not None and "delta_masked_minus_raw" in pivot.columns:
                for prop_name, delta in pivot["delta_masked_minus_raw"].dropna().items():
                    comet_experiment.log_metric(
                        f"{SELECTED_MODEL}/comparison/{prop_name}/delta_masked_minus_raw",
                        float(delta),
                    )
finally:
    if comet_experiment is not None:
        comet_experiment.end()


In [ ]:
# Optional: run all models on the same pipeline
RUN_MULTI_MODEL = False
MULTI_MODEL_KEYS = ["qwen3_vl_8b", "qwen2_5_vl_7b", "llava_onevision_1_5_8b"]

if RUN_MULTI_MODEL:
    multi_comet = init_comet_experiment(
        run_tag=f"multi_model_{DATASET_NAME}_{PROTOCOL_NAME}",
        run_params={
            "dataset_name": DATASET_NAME,
            "protocol_name": PROTOCOL_NAME,
            "model_keys": ",".join(MULTI_MODEL_KEYS),
            "num_samples": len(samples),
            "num_properties": len(property_specs),
            "include_only_gt_known": INCLUDE_ONLY_GT_KNOWN,
            "max_properties_per_sample": MAX_PROPERTIES_PER_SAMPLE,
            "prompt_mode": PROMPT_MODE,
            "property_batch_size": PROPERTY_BATCH_SIZE,
    "property_group_size": PROPERTY_GROUP_SIZE,
            "few_shot_k": FEW_SHOT_K,
    "few_shot_selection_mode": FEW_SHOT_SELECTION_MODE,
            "json_success_threshold": JSON_SUCCESS_THRESHOLD,
            "image_id_success_threshold": IMAGE_ID_SUCCESS_THRESHOLD,
        },
        enabled=COMET_ENABLED,
        default_project=COMET_DEFAULT_PROJECT,
    )

    try:
        comparison_df = run_many_models_safe(
            model_keys=MULTI_MODEL_KEYS,
            samples=samples,
            property_specs=property_specs,
            model_registry=MODEL_REGISTRY,
            reports_dir=REPORTS_DIR / PROTOCOL_NAME,
            variants=available_variants,
            prompt_mode=PROMPT_MODE,
            property_batch_size=PROPERTY_BATCH_SIZE,
            property_group_size=PROPERTY_GROUP_SIZE,
            few_shot_k=FEW_SHOT_K,
            few_shot_selection_mode=FEW_SHOT_SELECTION_MODE,
            json_success_threshold=JSON_SUCCESS_THRESHOLD,
            image_id_success_threshold=IMAGE_ID_SUCCESS_THRESHOLD,
            include_only_gt_known=INCLUDE_ONLY_GT_KNOWN,
            max_properties_per_sample=MAX_PROPERTIES_PER_SAMPLE,
            mask_background_mode=MASK_BACKGROUND_MODE,
            save_raw_output=SAVE_RAW_OUTPUT,
            comet_experiment=multi_comet,
        )
        display(comparison_df)
    finally:
        if multi_comet is not None:
            multi_comet.end()
else:
    print("Skip multi-model run. Set RUN_MULTI_MODEL=True to execute.")
